# Feature Engineering
**Input:** `processed/train_cleaned_for_good.csv` + `processed/test_cleaned_for_good.csv`  
**Output:** `processed/train_feat_good.csv` + `processed/test_feat_good.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
sns.set_style('whitegrid')

CLEAN_TRAIN = '../../processed/train_cleaned_for_good.csv'
CLEAN_TEST  = '../../processed/test_cleaned_for_good.csv'
OUT_TRAIN   = '../../processed/train_feat_good.csv'
OUT_TEST    = '../../processed/test_feat_good.csv'

df_train = pd.read_csv(CLEAN_TRAIN)
df_test  = pd.read_csv(CLEAN_TEST)
print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')
df_train.head(3)

## 1. Timestamp → Time Features

In [ ]:
def add_time_features(df):
    # Parse 'H:MM' → numeric time features
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour']         = parts[0]
    df['minute']       = parts[1]
    df['time_minutes'] = df['hour'] * 60 + df['minute']   # 0–1439

    # Cyclical encoding so 23:59 and 0:00 are close
    df['hour_sin']  = np.sin(2 * np.pi * df['hour']        / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour']        / 24)
    df['time_sin']  = np.sin(2 * np.pi * df['time_minutes']/ 1440)
    df['time_cos']  = np.cos(2 * np.pi * df['time_minutes']/ 1440)

    # Broad time-of-day buckets
    bins   = [-1, 5, 9, 12, 17, 20, 23]
    labels = ['night', 'morning_rush', 'midday', 'afternoon_rush', 'evening', 'late_night']
    df['time_of_day'] = pd.cut(df['hour'], bins=bins, labels=labels)

    # Peak flag
    df['is_peak'] = df['hour'].between(7, 9) | df['hour'].between(16, 19)
    df['is_peak'] = df['is_peak'].astype(int)

    return df

df_train = add_time_features(df_train)
df_test  = add_time_features(df_test)
print('Time features added:', ['hour','minute','time_minutes','hour_sin','hour_cos','time_sin','time_cos','time_of_day','is_peak'])

In [ ]:
# Verify cyclical encoding visually
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sample = df_train.drop_duplicates('hour').sort_values('hour')
axes[0].plot(sample['hour'], sample['hour_sin'], label='sin', marker='o')
axes[0].plot(sample['hour'], sample['hour_cos'], label='cos', marker='s')
axes[0].set_title('Cyclical Hour Encoding')
axes[0].legend()

# Demand by time of day
tod_demand = df_train.groupby('time_of_day', observed=True)['demand'].mean().sort_values(ascending=False)
axes[1].bar(tod_demand.index, tod_demand.values, color='steelblue')
axes[1].set_title('Mean Demand by Time-of-Day Bucket')
axes[1].set_ylabel('Mean Demand')
plt.tight_layout()
plt.show()

## 2. Geohash Features

In [ ]:
# Geohash prefix length — coarser spatial grain
df_train['geo4'] = df_train['geohash'].str[:4]
df_train['geo3'] = df_train['geohash'].str[:3]
df_test['geo4']  = df_test['geohash'].str[:4]
df_test['geo3']  = df_test['geohash'].str[:3]

# Historical mean / std demand per geohash (target statistics — computed on train only)
geo_stats = (
    df_train.groupby('geohash')['demand']
            .agg(geo_demand_mean='mean', geo_demand_std='std', geo_demand_median='median')
            .reset_index()
)
geo_stats['geo_demand_std'] = geo_stats['geo_demand_std'].fillna(0)

df_train = df_train.merge(geo_stats, on='geohash', how='left')
df_test  = df_test.merge(geo_stats,  on='geohash', how='left')

# Unseen geohashes in test → fill with global train mean
global_mean = df_train['demand'].mean()
global_std  = df_train['demand'].std()
global_med  = df_train['demand'].median()
df_test['geo_demand_mean']   = df_test['geo_demand_mean'].fillna(global_mean)
df_test['geo_demand_std']    = df_test['geo_demand_std'].fillna(global_std)
df_test['geo_demand_median'] = df_test['geo_demand_median'].fillna(global_med)

print('Geo target-stat columns added.')
df_train[['geohash','geo_demand_mean','geo_demand_std','geo_demand_median']].head()

In [ ]:
# geo4 target stats
geo4_stats = (
    df_train.groupby('geo4')['demand']
            .agg(geo4_demand_mean='mean', geo4_demand_std='std')
            .reset_index()
)
geo4_stats['geo4_demand_std'] = geo4_stats['geo4_demand_std'].fillna(0)

df_train = df_train.merge(geo4_stats, on='geo4', how='left')
df_test  = df_test.merge(geo4_stats,  on='geo4', how='left')
df_test['geo4_demand_mean'] = df_test['geo4_demand_mean'].fillna(global_mean)
df_test['geo4_demand_std']  = df_test['geo4_demand_std'].fillna(global_std)

print(f'geo4 unique train: {df_train["geo4"].nunique()}  | test: {df_test["geo4"].nunique()}')

## 3. Road & Infrastructure Features

In [ ]:
# Binary flags
df_train['is_highway']     = (df_train['RoadType'] == 'Highway').astype(int)
df_train['is_residential'] = (df_train['RoadType'] == 'Residential').astype(int)
df_train['large_vehicles_allowed'] = (df_train['LargeVehicles'] == 'Allowed').astype(int)
df_train['has_landmark']   = (df_train['Landmarks'] == 'Yes').astype(int)

df_test['is_highway']     = (df_test['RoadType'] == 'Highway').astype(int)
df_test['is_residential'] = (df_test['RoadType'] == 'Residential').astype(int)
df_test['large_vehicles_allowed'] = (df_test['LargeVehicles'] == 'Allowed').astype(int)
df_test['has_landmark']   = (df_test['Landmarks'] == 'Yes').astype(int)

# Road capacity proxy: highway counts extra
df_train['road_capacity'] = df_train['NumberofLanes'] * (1 + 0.5 * df_train['is_highway'])
df_test['road_capacity']  = df_test['NumberofLanes']  * (1 + 0.5 * df_test['is_highway'])

print('Road & infrastructure features added.')

## 4. Weather Features

In [ ]:
# Ordinal weather severity (Sunny=0, Foggy=1, Rainy=2, Snowy=3)
weather_severity = {'Sunny': 0, 'Foggy': 1, 'Rainy': 2, 'Snowy': 3}
df_train['weather_severity'] = df_train['Weather'].map(weather_severity)
df_test['weather_severity']  = df_test['Weather'].map(weather_severity)

# Bad weather flag
df_train['bad_weather'] = (df_train['weather_severity'] >= 2).astype(int)
df_test['bad_weather']  = (df_test['weather_severity']  >= 2).astype(int)

# Temperature bucket
temp_bins   = [-np.inf, 0, 10, 20, 30, np.inf]
temp_labels = ['freezing', 'cold', 'mild', 'warm', 'hot']
df_train['temp_bucket'] = pd.cut(df_train['Temperature'], bins=temp_bins, labels=temp_labels)
df_test['temp_bucket']  = pd.cut(df_test['Temperature'],  bins=temp_bins, labels=temp_labels)

# Temperature deviation from median (anomaly signal)
median_temp = df_train['Temperature'].median()
df_train['temp_deviation'] = (df_train['Temperature'] - median_temp).abs()
df_test['temp_deviation']  = (df_test['Temperature']  - median_temp).abs()

print('Weather features added.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

weather_demand = df_train.groupby('Weather')['demand'].mean().sort_values()
axes[0].barh(weather_demand.index, weather_demand.values, color='steelblue')
axes[0].set_title('Mean Demand by Weather')

axes[1].scatter(df_train['temp_deviation'].sample(3000, random_state=42),
                df_train['demand'].sample(3000, random_state=42),
                alpha=0.2, s=5, color='coral')
axes[1].set_title('Temp Deviation vs Demand')
axes[1].set_xlabel('|Temperature - median|')

plt.tight_layout()
plt.show()

## 5. Interaction Features

In [ ]:
# Peak × bad weather — rush hour in poor conditions
df_train['peak_bad_weather'] = df_train['is_peak'] * df_train['bad_weather']
df_test['peak_bad_weather']  = df_test['is_peak']  * df_test['bad_weather']

# Highway × NumberofLanes (wider highways)
df_train['highway_lanes'] = df_train['is_highway'] * df_train['NumberofLanes']
df_test['highway_lanes']  = df_test['is_highway']  * df_test['NumberofLanes']

# Landmark × peak (points of interest at rush hour)
df_train['landmark_peak'] = df_train['has_landmark'] * df_train['is_peak']
df_test['landmark_peak']  = df_test['has_landmark']  * df_test['is_peak']

# Temperature × weather_severity
df_train['temp_x_severity'] = df_train['Temperature'] * df_train['weather_severity']
df_test['temp_x_severity']  = df_test['Temperature']  * df_test['weather_severity']

print('Interaction features added.')

## 6. Lag / Historical Demand Features (train only)

In [ ]:
# Sort to ensure correct order for lags
df_train = df_train.sort_values(['geohash', 'day', 'time_minutes']).reset_index(drop=True)

# Rolling mean demand per geohash over the last N time slots
for w in [3, 6, 12]:
    col = f'geo_demand_roll{w}'
    df_train[col] = (
        df_train.groupby('geohash')['demand']
                .transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
    )

# Lag 1 (previous time slot demand per geohash)
df_train['geo_demand_lag1'] = (
    df_train.groupby('geohash')['demand']
            .transform(lambda x: x.shift(1))
)

# For test set, use the full-geohash mean (no future leakage risk)
for w in [3, 6, 12]:
    df_test[f'geo_demand_roll{w}'] = df_test['geohash'].map(geo_stats.set_index('geohash')['geo_demand_mean']).fillna(global_mean)
df_test['geo_demand_lag1'] = df_test['geohash'].map(geo_stats.set_index('geohash')['geo_demand_mean']).fillna(global_mean)

# Fill NaN lag values in train (first records per geohash)
for col in ['geo_demand_roll3','geo_demand_roll6','geo_demand_roll12','geo_demand_lag1']:
    df_train[col] = df_train[col].fillna(df_train['geo_demand_mean'])

print('Lag/rolling features added.')
df_train[['geohash','demand','geo_demand_lag1','geo_demand_roll3','geo_demand_roll6']].head()

## 7. Label Encoding for Categoricals

In [ ]:
from sklearn.preprocessing import LabelEncoder

cat_encode = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'time_of_day', 'temp_bucket']

for col in cat_encode:
    le = LabelEncoder()
    combined = pd.concat([df_train[col].astype(str), df_test[col].astype(str)], ignore_index=True)
    le.fit(combined)
    df_train[col + '_enc'] = le.transform(df_train[col].astype(str))
    df_test[col  + '_enc'] = le.transform(df_test[col].astype(str))

# Geohash frequency encoding
geo_freq = df_train['geohash'].value_counts(normalize=True)
df_train['geo_freq'] = df_train['geohash'].map(geo_freq).fillna(0)
df_test['geo_freq']  = df_test['geohash'].map(geo_freq).fillna(0)

geo4_freq = df_train['geo4'].value_counts(normalize=True)
df_train['geo4_freq'] = df_train['geo4'].map(geo4_freq).fillna(0)
df_test['geo4_freq']  = df_test['geo4'].map(geo4_freq).fillna(0)

print('Label + frequency encoding done.')

## 8. Feature Summary & Correlation with Target

In [ ]:
num_feature_cols = [
    'hour', 'minute', 'time_minutes', 'hour_sin', 'hour_cos', 'time_sin', 'time_cos',
    'is_peak', 'is_highway', 'is_residential', 'large_vehicles_allowed', 'has_landmark',
    'road_capacity', 'NumberofLanes',
    'weather_severity', 'bad_weather', 'temp_deviation',
    'peak_bad_weather', 'highway_lanes', 'landmark_peak', 'temp_x_severity',
    'geo_demand_mean', 'geo_demand_std', 'geo_demand_median',
    'geo4_demand_mean', 'geo4_demand_std',
    'geo_demand_lag1', 'geo_demand_roll3', 'geo_demand_roll6', 'geo_demand_roll12',
    'geo_freq', 'geo4_freq',
    'Temperature',
    'RoadType_enc', 'LargeVehicles_enc', 'Landmarks_enc', 'Weather_enc',
    'time_of_day_enc', 'temp_bucket_enc',
]

corrs = df_train[num_feature_cols + ['demand']].corr()['demand'].drop('demand').sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 12))
corrs.plot(kind='barh', color=corrs.apply(lambda v: 'steelblue' if v >= 0 else 'coral'))
plt.title('Feature Correlation with demand')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

print(f'\nTotal features: {len(num_feature_cols)}')
print('\nTop 10 correlated features:')
print(corrs.head(10))

## 9. Save Feature-Engineered Data

In [ ]:
# Columns to keep: all features + identifiers + target
id_cols   = ['geohash', 'geo3', 'geo4', 'day', 'timestamp']
target    = ['demand']
keep_cols = id_cols + num_feature_cols

train_out = df_train[keep_cols + target].copy()
test_out  = df_test[keep_cols].copy()

train_out.to_csv(OUT_TRAIN, index=False)
test_out.to_csv(OUT_TEST,   index=False)

print(f'Saved: {OUT_TRAIN}  shape={train_out.shape}')
print(f'Saved: {OUT_TEST}   shape={test_out.shape}')
train_out.head(3)